# Insurance Fraud Detection — Bước 1: EDA & Imbalance Analysis
**Dataset:** Auto Insurance Fraud Claims (Kaggle)  
**Output:** `fraud_cleaned.csv` · `eda_summary.json`

---
### Cấu trúc notebook
1. Cài đặt & cấu hình
2. Load & kiểm tra dữ liệu thô
3. Phân tích biến mục tiêu — fraud distribution
4. Phân tích imbalance — tại sao Accuracy là metric sai
5. EDA biến categorical
6. EDA biến numerical
7. Correlation & multicollinearity
8. Phân tích theo thời gian (claim patterns)
9. Xử lý missing values & encoding sơ bộ
10. Lưu dữ liệu sạch & EDA summary


## Fraud Detection vs Credit Scoring — Tại sao khác nhau?

| Tiêu chí | Credit Scoring (Dự án 1) | Fraud Detection (Dự án 2) |
|---|---|---|
| Tỷ lệ sự kiện | 6.7% default | ~24% fraud |
| Metric chính | AUC / Gini / KS | PR-AUC / F1 / Recall |
| Model | Logistic Regression | XGBoost |
| Interpretability | Bắt buộc (Basel II) | SHAP (tự chọn) |
| Cost asymmetry | Balanced | Bắt fraud > Tránh false positive |
| Feature engineering | WOE binning | Free-form (thời gian, hành vi) |

### Tại sao Accuracy vô nghĩa với fraud?

Nếu model luôn predict "không fraud" → Accuracy = 76% (vì 76% claims là legit).  
Nhưng model đó **bắt được 0 fraud case** — hoàn toàn vô dụng.

**Metric đúng:**
- **Recall** = TP / (TP + FN) → Bắt được bao nhiêu % fraud thực sự?
- **Precision** = TP / (TP + FP) → Trong số bị flag, bao nhiêu % thực sự là fraud?
- **PR-AUC** = Area under Precision-Recall curve → Tổng hợp cả hai
- **F1** = Harmonic mean của Precision và Recall


## 1. Cài đặt & cấu hình

In [ ]:
import warnings, json
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency, pointbiserialr

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
pd.set_option("display.max_rows", 60)

plt.rcParams.update({
    "figure.dpi": 115,
    "figure.figsize": (12, 5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.titlepad": 10,
    "font.family": "sans-serif",
})

PALETTE = {
    "fraud":   "#E53935",
    "legit":   "#2196F3",
    "neutral": "#78909C",
    "warn":    "#FF8F00",
    "good":    "#43A047",
}

print("✓ Setup hoàn tất")


## 2. Load & kiểm tra dữ liệu thô

> **Download dataset:**
> 1. Vào https://www.kaggle.com/datasets/buntyshah/auto-insurance-claims-data
> 2. Download `insurance_claims.csv`
> 3. Đặt cùng thư mục với notebook này


In [ ]:
df_raw = pd.read_csv("insurance_claims.csv")

# Đổi tên target cho nhất quán
df_raw = df_raw.rename(columns={"fraud_reported": "target"})
df_raw["target"] = (df_raw["target"] == "Y").astype(int)

print(f"Shape  : {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
print(f"Memory : {df_raw.memory_usage(deep=True).sum() / 1024:.1f} KB")
print()

# Phân loại cột
NUM_COLS  = df_raw.select_dtypes(include=["int64","float64"]).columns.tolist()
CAT_COLS  = df_raw.select_dtypes(include=["object"]).columns.tolist()
NUM_COLS  = [c for c in NUM_COLS if c != "target"]

print(f"Numerical   ({len(NUM_COLS)}): {NUM_COLS}")
print(f"Categorical ({len(CAT_COLS)}): {CAT_COLS}")
print()
df_raw.head(3)


In [ ]:
# Tổng quan dtypes + sample values
print("── Thống kê mô tả numerical ──")
print(df_raw[NUM_COLS].describe(percentiles=[.01,.25,.5,.75,.99]).T.round(2).to_string())


## 3. Phân tích biến mục tiêu — Fraud Distribution

In [ ]:
fraud_counts = df_raw["target"].value_counts()
fraud_pct    = df_raw["target"].value_counts(normalize=True) * 100

print("Target distribution:")
print(f"  Legit (0) : {fraud_counts[0]:,}  ({fraud_pct[0]:.1f}%)")
print(f"  Fraud (1) : {fraud_counts[1]:,}  ({fraud_pct[1]:.1f}%)")
print(f"  Imbalance ratio: {fraud_counts[0]/fraud_counts[1]:.1f}:1")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Bar chart
bars = axes[0].bar(
    ["Legit (0)", "Fraud (1)"],
    fraud_counts.values,
    color=[PALETTE["legit"], PALETTE["fraud"]],
    width=0.5, edgecolor="white",
)
for bar, pct in zip(bars, fraud_pct.values):
    axes[0].text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 30,
        f"{pct:.1f}%", ha="center", fontsize=11, fontweight="bold",
    )
axes[0].set_title("Target Distribution")
axes[0].set_ylabel("Count")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x:,.0f}"))

# Pie
axes[1].pie(
    fraud_counts.values,
    labels=[f"Legit
{fraud_pct[0]:.1f}%", f"Fraud
{fraud_pct[1]:.1f}%"],
    colors=[PALETTE["legit"], PALETTE["fraud"]],
    startangle=90,
    wedgeprops={"edgecolor":"white","linewidth":2},
    textprops={"fontsize":10},
)
axes[1].set_title("Fraud vs Legit")

# Accuracy paradox demo
axes[2].barh(
    ["Dummy model
(always predict legit)", "Target
(our goal)"],
    [fraud_pct[0], 95],
    color=[PALETTE["warn"], PALETTE["good"]],
    height=0.4, edgecolor="white",
)
axes[2].axvline(80, color="gray", linestyle="--", lw=1, alpha=0.7)
for i, (val, label_) in enumerate(zip([fraud_pct[0], 95], ["Accuracy", "F1 target"])):
    axes[2].text(val + 0.5, i, f"{val:.1f}%", va="center", fontsize=10, fontweight="bold")
axes[2].set_xlim(0, 110)
axes[2].set_title("Accuracy Paradox")
axes[2].set_xlabel("Score (%)")

plt.suptitle("Fraud Distribution Analysis", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("fraud_01_target_dist.png", bbox_inches="tight")
plt.show()


## 4. Phân tích Imbalance — Chọn metric đúng

Với fraud rate ~24%, ta cần hiểu **cost asymmetry**:
- **Miss một fraud case (FN):** Công ty bảo hiểm trả tiền oan → thiệt hại trực tiếp
- **Flag nhầm legit case (FP):** Khách hàng tốt bị điều tra → mất uy tín, customer churn

Tùy chiến lược kinh doanh, cost của FN thường cao hơn FP 5–10x.
→ Ta sẽ tối ưu **Recall** (bắt nhiều fraud) nhưng giữ Precision ở mức chấp nhận được.


In [ ]:
# ── Demo: So sánh metric khi dùng dummy model ────────────────────────────────
from sklearn.metrics import (accuracy_score, f1_score, recall_score,
                              precision_score, average_precision_score,
                              confusion_matrix)
from sklearn.dummy import DummyClassifier

y = df_raw["target"]

# Dummy model: always predict majority class
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(y.values.reshape(-1,1), y)
y_dummy = dummy.predict(y.values.reshape(-1,1))

print("═══════════════════════════════════════════")
print("  Dummy model (luôn predict 'legit')")
print("═══════════════════════════════════════════")
print(f"  Accuracy  : {accuracy_score(y, y_dummy)*100:.2f}%  ← Trông có vẻ tốt!")
print(f"  Recall    : {recall_score(y, y_dummy)*100:.2f}%    ← Bắt được 0 fraud!")
print(f"  Precision : {precision_score(y, y_dummy, zero_division=0)*100:.2f}%")
print(f"  F1 Score  : {f1_score(y, y_dummy)*100:.2f}%")
print()

# ── Cost matrix analysis ──────────────────────────────────────────────────────
avg_claim   = 45_000     # USD — giả sử claim trung bình
invest_cost = 500        # Chi phí điều tra một case

fraud_cases  = fraud_counts[1]
legit_cases  = fraud_counts[0]

fn_cost = avg_claim       # Trả tiền oan cho fraud
fp_cost = invest_cost     # Điều tra nhầm legit

print("Cost Matrix Analysis (giả định):")
print(f"  Average fraud claim    : ${avg_claim:,.0f}")
print(f"  Investigation cost     : ${invest_cost:,.0f}")
print(f"  Cost ratio FN/FP       : {fn_cost/fp_cost:.0f}x")
print()
print(f"  Nếu miss tất cả {fraud_cases} fraud cases:")
print(f"  → Thiệt hại: ${fraud_cases * avg_claim:,.0f}")
print()
print(f"  Nếu điều tra tất cả {legit_cases} legit cases nhầm:")
print(f"  → Chi phí: ${legit_cases * invest_cost:,.0f}")
print()
print("→ Kết luận: Recall quan trọng hơn Precision ~{:.0f}x trong bài toán này".format(
    fn_cost/fp_cost
))


## 5. EDA Biến Categorical — Fraud Rate theo từng nhóm

Với mỗi biến categorical, ta tính **fraud rate** trong từng nhóm.
Những nhóm có fraud rate cao bất thường → feature quan trọng.

Dùng **Chi-square test** để kiểm tra xem sự khác biệt có ý nghĩa thống kê không.


In [ ]:
def fraud_rate_by_category(df, col, target="target", top_n=10):
    """
    Tính fraud rate theo từng giá trị của cột categorical.
    Kèm chi-square test để kiểm tra significance.
    """
    grp = (df.groupby(col)[target]
             .agg(["count","mean"])
             .rename(columns={"count":"n","mean":"fraud_rate"})
             .sort_values("fraud_rate", ascending=False)
             .head(top_n))
    grp["fraud_rate_pct"] = grp["fraud_rate"] * 100

    # Chi-square test
    ct = pd.crosstab(df[col], df[target])
    chi2, p, _, _ = chi2_contingency(ct)
    grp.attrs["chi2"] = chi2
    grp.attrs["p_value"] = p
    return grp

# ── Chọn categorical cols quan trọng để visualize ────────────────────────────
CAT_IMPORTANT = [
    "incident_type", "incident_severity", "authorities_contacted",
    "insured_education_level", "insured_occupation", "policy_state",
    "auto_make", "insured_hobbies", "collision_type", "property_damage",
]
# Lọc chỉ những cột thực sự tồn tại trong dataset
CAT_IMPORTANT = [c for c in CAT_IMPORTANT if c in df_raw.columns]

print("Fraud rate & Chi-square test theo categorical features:")
print("─" * 65)
print(f"{'Column':30s} {'Top fraud group':20s} {'Fraud rate':>11} {'p-value':>10}")
print("─" * 65)

cat_stats = {}
for col in CAT_IMPORTANT:
    grp = fraud_rate_by_category(df_raw, col)
    top_group = grp.index[0]
    top_rate  = grp.iloc[0]["fraud_rate_pct"]
    p_val     = grp.attrs["p_value"]
    sig       = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
    print(f"  {col:28s} {str(top_group):20s} {top_rate:>9.1f}%  {p_val:>8.4f} {sig}")
    cat_stats[col] = grp


In [ ]:
# ── Visualize top 6 categorical features ─────────────────────────────────────
PLOT_CATS = CAT_IMPORTANT[:6]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
overall_rate = df_raw["target"].mean() * 100

for i, col in enumerate(PLOT_CATS):
    grp = cat_stats[col].sort_values("fraud_rate_pct", ascending=True)
    bar_colors = [
        PALETTE["fraud"] if r > overall_rate * 1.3 else
        PALETTE["legit"] if r < overall_rate * 0.7 else
        PALETTE["neutral"]
        for r in grp["fraud_rate_pct"]
    ]
    bars = axes[i].barh(
        grp.index.astype(str),
        grp["fraud_rate_pct"],
        color=bar_colors, height=0.55, edgecolor="white",
    )
    for bar, (_, row) in zip(bars, grp.iterrows()):
        axes[i].text(
            bar.get_width() + 0.3,
            bar.get_y() + bar.get_height()/2,
            f"{row['fraud_rate_pct']:.1f}% (n={row['n']:,})",
            va="center", fontsize=8,
        )
    axes[i].axvline(overall_rate, color="gray", linestyle="--",
                    lw=1.2, alpha=0.7, label=f"Overall {overall_rate:.1f}%")
    axes[i].set_xlabel("Fraud Rate (%)")
    p_val = cat_stats[col].attrs["p_value"]
    sig_str = "p<0.001 ***" if p_val < 0.001 else f"p={p_val:.3f}"
    axes[i].set_title(f"{col}
({sig_str})", fontsize=10)
    axes[i].legend(fontsize=8, frameon=False)
    axes[i].set_xlim(0, grp["fraud_rate_pct"].max() * 1.35)

plt.suptitle("Fraud Rate theo Categorical Features",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("fraud_02_categorical_eda.png", bbox_inches="tight")
plt.show()


## 6. EDA Biến Numerical — Distribution & Point-Biserial Correlation

**Point-biserial correlation** đo mức độ tương quan giữa biến số và biến nhị phân (fraud/legit).
Tương đương Pearson correlation nhưng áp dụng cho biến binary.


In [ ]:
# ── Point-biserial correlation ───────────────────────────────────────────────
pb_corr = {}
for col in NUM_COLS:
    clean = df_raw[[col, "target"]].dropna()
    if clean[col].std() > 0:
        corr, p_val = pointbiserialr(clean["target"], clean[col])
        pb_corr[col] = {"correlation": corr, "p_value": p_val,
                        "abs_corr": abs(corr)}

pb_df = (pd.DataFrame(pb_corr).T
           .sort_values("abs_corr", ascending=False)
           .reset_index()
           .rename(columns={"index":"Variable"}))

print("Point-Biserial Correlation với target (fraud):")
print(pb_df[["Variable","correlation","p_value"]].to_string(index=False))

# Visualize
fig, ax = plt.subplots(figsize=(9, max(4, len(pb_df)*0.45)))
colors_corr = [
    PALETTE["fraud"] if c > 0 else PALETTE["legit"]
    for c in pb_df["correlation"]
]
bars = ax.barh(
    pb_df["Variable"][::-1],
    pb_df["correlation"][::-1],
    color=colors_corr[::-1],
    height=0.55, edgecolor="white",
)
for bar, (_, row) in zip(bars, pb_df[::-1].iterrows()):
    sig = "***" if row["p_value"]<0.001 else "**" if row["p_value"]<0.01 else "*" if row["p_value"]<0.05 else ""
    xpos = bar.get_width() + 0.003 if bar.get_width() >= 0 else bar.get_width() - 0.003
    ha   = "left" if bar.get_width() >= 0 else "right"
    ax.text(xpos, bar.get_y() + bar.get_height()/2,
            f"{row['correlation']:.3f} {sig}",
            va="center", ha=ha, fontsize=8.5)
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("Point-Biserial Correlation")
ax.set_title("Numerical Features — Correlation với Fraud")
plt.tight_layout()
plt.savefig("fraud_03_numerical_correlation.png", bbox_inches="tight")
plt.show()


In [ ]:
# ── Distribution plots: Fraud vs Legit ───────────────────────────────────────
# Chỉ plot top 8 biến có correlation cao nhất
top_num = pb_df.head(min(8, len(pb_df)))["Variable"].tolist()

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()

for i, col in enumerate(top_num):
    legit_vals = df_raw.loc[df_raw["target"]==0, col].dropna()
    fraud_vals = df_raw.loc[df_raw["target"]==1, col].dropna()

    axes[i].hist(legit_vals, bins=40, alpha=0.6, color=PALETTE["legit"],
                 label="Legit", density=True, edgecolor="none")
    axes[i].hist(fraud_vals, bins=40, alpha=0.6, color=PALETTE["fraud"],
                 label="Fraud", density=True, edgecolor="none")

    axes[i].axvline(legit_vals.mean(), color=PALETTE["legit"],
                    linestyle="--", lw=1.5, alpha=0.8)
    axes[i].axvline(fraud_vals.mean(), color=PALETTE["fraud"],
                    linestyle="--", lw=1.5, alpha=0.8)

    # KS test
    ks_stat, ks_p = stats.ks_2samp(legit_vals, fraud_vals)
    axes[i].set_title(
        f"{col}
KS={ks_stat:.3f}, p={'<0.001' if ks_p<0.001 else f'{ks_p:.3f}'}",
        fontsize=9,
    )
    axes[i].legend(fontsize=8, frameon=False)
    axes[i].tick_params(labelsize=8)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Distribution: Fraud vs Legit (top correlated features)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("fraud_04_num_distributions.png", bbox_inches="tight")
plt.show()


## 7. Phân tích Temporal Patterns — Dấu hiệu thời gian của Fraud

Fraud thường có **patterns theo thời gian** đặc trưng:
- Claim được nộp ngay sau khi mua policy (policy age rất ngắn)
- Incident xảy ra vào cuối tuần hoặc đêm (ít nhân chứng hơn)
- Claim delay bất thường (nộp quá muộn hoặc quá nhanh)


In [ ]:
df = df_raw.copy()

# ── Parse dates nếu có ───────────────────────────────────────────────────────
date_cols = [c for c in df.columns if "date" in c.lower()]
print(f"Date columns found: {date_cols}")

for col in date_cols:
    try:
        df[col] = pd.to_datetime(df[col], errors="coerce")
        print(f"  Parsed: {col} → {df[col].dtype}")
    except Exception as e:
        print(f"  Could not parse {col}: {e}")

# ── Policy age (months) at time of incident ───────────────────────────────────
if "policy_bind_date" in df.columns and "incident_date" in df.columns:
    df["policy_age_days"] = (
        df["incident_date"] - df["policy_bind_date"]
    ).dt.days
    print(f"\npolicy_age_days created:")
    print(df.groupby("target")["policy_age_days"].describe().round(1))

    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for t, lbl, col_ in [(0,"Legit",PALETTE["legit"]),(1,"Fraud",PALETTE["fraud"])]:
        vals = df.loc[df["target"]==t, "policy_age_days"].dropna()
        axes[0].hist(vals, bins=50, alpha=0.6, color=col_,
                     label=lbl, density=True, edgecolor="none")
    axes[0].set_xlabel("Policy age at incident (days)")
    axes[0].set_title("Policy Age Distribution")
    axes[0].legend(frameon=False)

    # Incident by month
    if not df["incident_date"].isna().all():
        df["incident_month"] = df["incident_date"].dt.month
        monthly = df.groupby(["incident_month","target"])["target"].count().unstack()
        if 0 in monthly.columns and 1 in monthly.columns:
            monthly["fraud_rate"] = monthly[1] / (monthly[0] + monthly[1]) * 100
            axes[1].bar(monthly.index, monthly["fraud_rate"],
                        color=PALETTE["fraud"], alpha=0.8, edgecolor="white")
            axes[1].axhline(df["target"].mean()*100, color="gray",
                            linestyle="--", lw=1.2, label="Overall rate")
            axes[1].set_xlabel("Month")
            axes[1].set_ylabel("Fraud rate (%)")
            axes[1].set_title("Fraud Rate by Month")
            axes[1].legend(frameon=False)

    plt.tight_layout()
    plt.savefig("fraud_05_temporal.png", bbox_inches="tight")
    plt.show()
else:
    print("Không tìm thấy date columns — bỏ qua temporal analysis")
    print("→ Sẽ tạo synthetic time features ở Bước 2 (Feature Engineering)")


## 8. Phân tích Missing Values

In [ ]:
def missing_summary(df):
    total   = df.isnull().sum()
    pct     = total / len(df) * 100
    dtype   = df.dtypes
    summary = pd.DataFrame({
        "Missing count": total,
        "Missing %":     pct.round(2),
        "Dtype":         dtype,
    }).sort_values("Missing %", ascending=False)
    return summary[summary["Missing count"] > 0]

miss = missing_summary(df_raw)
if len(miss) == 0:
    print("✓ Không có missing values trong dataset")
else:
    print(f"Số cột có missing: {len(miss)}")
    print(miss.to_string())

# ── Kiểm tra ký tự "?" thường dùng thay cho NaN trong insurance data ─────────
question_mark_cols = []
for col in CAT_COLS:
    if "?" in df_raw[col].astype(str).values:
        n_q = (df_raw[col].astype(str) == "?").sum()
        question_mark_cols.append((col, n_q))
        print(f"  [{col}] có {n_q} giá trị '?' — cần replace với NaN")

if not question_mark_cols:
    print("✓ Không có ký tự '?' trong categorical columns")


## 9. Xử lý Missing Values & Làm sạch sơ bộ

In [ ]:
df = df_raw.copy()

# ── 1. Replace '?' với NaN ────────────────────────────────────────────────────
for col in CAT_COLS:
    df[col] = df[col].replace("?", np.nan)

n_nan_after = df.isnull().sum().sum()
print(f"NaN sau khi replace '?': {n_nan_after}")

# ── 2. Drop cột không có giá trị dự báo ──────────────────────────────────────
# policy_number: ID duy nhất — không có predictive value
# _c39 hoặc cột unnamed nếu có
DROP_COLS = []
for col in df.columns:
    if col.lower() in ["policy_number", "unnamed: 0", "_c39"]:
        DROP_COLS.append(col)
    elif df[col].nunique() == len(df):   # Tất cả unique → likely ID column
        DROP_COLS.append(col)

if DROP_COLS:
    df = df.drop(columns=DROP_COLS)
    print(f"Dropped ID/useless columns: {DROP_COLS}")

# ── 3. Xử lý '?' còn lại trong categorical ───────────────────────────────────
for col in [c for c in df.columns if df[c].dtype == "object" and c != "target"]:
    n_null = df[col].isnull().sum()
    if n_null > 0:
        mode_val = df[col].mode()[0]
        df[col] = df[col].fillna(mode_val)
        print(f"  {col}: filled {n_null} NaN với mode='{mode_val}'")

# ── 4. Xử lý numerical missing ───────────────────────────────────────────────
for col in NUM_COLS:
    n_null = df[col].isnull().sum()
    if n_null > 0:
        med = df[col].median()
        df[col] = df[col].fillna(med)
        print(f"  {col}: filled {n_null} NaN với median={med:.2f}")

print(f"\n✓ Missing values sau xử lý: {df.isnull().sum().sum()}")
print(f"✓ Shape: {df.shape}")


## 10. Outlier Check cho Numerical Features

**Lưu ý quan trọng:** Trong fraud detection, một số outlier có thể là **dấu hiệu fraud** thực sự
(ví dụ: claim amount cực cao, số lần claim nhiều bất thường).
→ Không winsorize tất cả — cần phân tích trước khi quyết định.


In [ ]:
def outlier_report(df, cols):
    rows = []
    for col in cols:
        s = df[col].dropna()
        q1, q3 = s.quantile([0.25, 0.75])
        iqr    = q3 - q1
        lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
        n_out  = ((s < lo) | (s > hi)).sum()

        # Fraud rate trong outlier group
        out_mask    = (df[col] < lo) | (df[col] > hi)
        fraud_in_out = df.loc[out_mask, "target"].mean() * 100 if out_mask.sum() > 0 else 0
        fraud_in_norm= df.loc[~out_mask, "target"].mean() * 100

        rows.append({
            "Column":              col,
            "N outliers":          n_out,
            "Outlier %":           round(n_out/len(s)*100, 2),
            "Fraud % in outliers": round(fraud_in_out, 1),
            "Fraud % normal":      round(fraud_in_norm, 1),
            "Delta":               round(fraud_in_out - fraud_in_norm, 1),
        })
    return (pd.DataFrame(rows)
              .sort_values("Delta", ascending=False)
              .reset_index(drop=True))

num_present = [c for c in NUM_COLS if c in df.columns]
out_df = outlier_report(df, num_present)
print("Outlier analysis — Fraud rate trong outlier group vs normal:")
print(out_df.to_string(index=False))

print()
print("Columns với outlier fraud rate cao hơn normal (keep as features!):")
keep_outlier = out_df[out_df["Delta"] > 5]
for _, row in keep_outlier.iterrows():
    print(f"  {row['Column']:30s} Delta = +{row['Delta']:.1f}%  → Có giá trị dự báo")


## 11. Lưu dữ liệu sạch & EDA Summary

In [ ]:
# ── Final check ──────────────────────────────────────────────────────────────
print("═══ KIỂM TRA CHẤT LƯỢNG DỮ LIỆU SAU XỬ LÝ ═══")
print(f"  Shape          : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Missing values : {df.isnull().sum().sum()}")
print(f"  Fraud cases    : {df['target'].sum():,} ({df['target'].mean()*100:.1f}%)")
print(f"  Legit cases    : {(df['target']==0).sum():,}")
print(f"  Imbalance ratio: {(df['target']==0).sum()/df['target'].sum():.1f}:1")

# ── Save cleaned data ─────────────────────────────────────────────────────────
df.to_csv("fraud_cleaned.csv", index=False)
print("\n✓ Saved: fraud_cleaned.csv")

# ── Save EDA summary JSON ─────────────────────────────────────────────────────
eda_summary = {
    "n_records":        int(df.shape[0]),
    "n_features":       int(df.shape[1] - 1),
    "fraud_count":      int(df["target"].sum()),
    "legit_count":      int((df["target"]==0).sum()),
    "fraud_rate":       round(float(df["target"].mean()), 4),
    "imbalance_ratio":  round(float((df["target"]==0).sum() / df["target"].sum()), 2),
    "n_categorical":    int(df.select_dtypes("object").shape[1]),
    "n_numerical":      int(df.select_dtypes(["int64","float64"]).shape[1] - 1),
    "top_cat_features": [
        {"column": col, "p_value": round(cat_stats[col].attrs["p_value"], 6)}
        for col in list(cat_stats.keys())[:5]
        if col in cat_stats and cat_stats[col].attrs["p_value"] < 0.05
    ],
    "top_num_features": pb_df.head(5)[["Variable","correlation"]].to_dict("records"),
    "missing_after_clean": int(df.isnull().sum().sum()),
}
with open("eda_summary.json", "w") as f:
    json.dump(eda_summary, f, indent=2)
print("✓ Saved: eda_summary.json")

print()
print("═══ TÓM TẮT BƯỚC 1 ═══")
print(f"  Fraud rate        : {df['target'].mean()*100:.1f}%")
print(f"  Imbalance ratio   : {(df['target']==0).sum()/df['target'].sum():.1f}:1")
print(f"  Top categorical   : {list(cat_stats.keys())[:3]}")
print(f"  Top numerical     : {pb_df.head(3)['Variable'].tolist()}")
print()
print("─── Bước tiếp theo: Bước 2 — Feature Engineering ───")
